# Processamento dos dados brutos

Este notebook transforma os arquivos JSON coletados da API do Chess.com em arquivos Parquet prontos para análise. A etapa aplica filtros básicos de qualidade, remove colunas que não serão usadas no EDA inicial e consolida os dados por modalidade.

O objetivo aqui é produzir uma base mais leve e consistente para o notebook de EDA, mantendo apenas partidas rated com regras padrão de xadrez.


### 1. Critérios de limpeza

A base bruta contém todas as informações retornadas pela API para cada partida coletada. Nesta etapa, aplico uma limpeza inicial com três decisões principais:

- remover colunas pouco úteis para o EDA inicial;
- manter apenas partidas rated;
- manter apenas partidas com `rules == "chess"`, excluindo variantes.


As colunas removidas são `url`, `tcn`, `initial_setup` e `tournament`.

Essas variáveis não entram nas primeiras análises e aumentam o tamanho dos arquivos. Campos mais relevantes para a hipótese, como ratings, resultado, PGN, entre outros, são preservados.


In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath(".."))

In [2]:
from pathlib import Path

pasta_base_raw = Path("../data/raw")
pasta_base_processed = Path("../data/processed")

time_class = ["bullet", "blitz", "rapid"]
cols_to_drop = ['url', 'tcn', 'initial_setup', 'tournament']



# Percorre os tipos de jogo
for tipo in time_class:
    dfs = list()    
    pasta_entrada = pasta_base_raw / tipo
    pasta_saida = pasta_base_processed / tipo
    
    if not pasta_entrada.exists():
        print(f"Pasta não encontrada: {pasta_entrada}. Pulando...")
        continue
        
    pasta_saida.mkdir(parents=True, exist_ok=True)
    
    print(f"\nProcessando arquivos de: {tipo}...")
    
    for arquivo_json in pasta_entrada.glob("*.json"):
        
        df = pd.read_json(arquivo_json)
        df = df.drop(columns=cols_to_drop, errors="ignore")
        df = df.loc[df["rated"] == True]
        df = df.loc[df["rules"] == "chess"]

        # Pega o nome do arquivo e troca a extensão
        nome_arquivo = arquivo_json.with_suffix(".parquet").name
        
        arquivo_parquet = pasta_saida / nome_arquivo
        
        df.to_parquet(arquivo_parquet, index=False)
        print(f"   Salvo: {arquivo_parquet}")

        # DF concatenado
        df["time_control"] = df["time_control"].astype(str)
        dfs.append(df)
    
    df_concat = pd.concat(dfs, ignore_index=True)
    df_concat.to_parquet(pasta_saida / f"top_50_{tipo}.parquet")



Processando arquivos de: bullet...
   Salvo: ../data/processed/bullet/vi_pranav_2026-05-26_16-14-58.parquet
   Salvo: ../data/processed/bullet/hikaru_2026-05-26_16-09-32.parquet
   Salvo: ../data/processed/bullet/ChristianArca_2026-05-26_16-13-10.parquet
   Salvo: ../data/processed/bullet/MagnusCarlsen_2026-05-26_16-12-24.parquet
   Salvo: ../data/processed/bullet/WorriedEgg_2026-05-26_16-11-30.parquet
   Salvo: ../data/processed/bullet/Polish_fighter3000_2026-05-26_16-15-32.parquet
   Salvo: ../data/processed/bullet/OhanyanEminChess_2026-05-26_16-13-29.parquet
   Salvo: ../data/processed/bullet/Konavets_2026-05-26_16-13-54.parquet
   Salvo: ../data/processed/bullet/Jospem_2026-05-26_16-12-42.parquet
   Salvo: ../data/processed/bullet/ChessLover0108_2026-05-26_16-13-05.parquet
   Salvo: ../data/processed/bullet/Dolphin_2010_2026-05-26_16-14-15.parquet
   Salvo: ../data/processed/bullet/FairChess_on_YouTube_2026-05-26_16-12-09.parquet
   Salvo: ../data/processed/bullet/penguingm1_2026-

### 2. Saídas geradas

Para cada modalidade, o processamento salva dois tipos de arquivo em `data/processed`:

- um arquivo Parquet para cada jogador processado;
- um arquivo consolidado `top_50_{modalidade}.parquet`, usado no EDA.

Depois desta etapa, o notebook de EDA pode carregar diretamente os três arquivos consolidados de bullet, blitz e rapid.
